# EDA: Khám phá Mock Data Cà Phê (AI002)

**Mục tiêu:** Khám phá cấu trúc dữ liệu, phân tích tương quan, phát hiện outliers và mùa vụ.

> **Trụ cột Bias (2):** Dữ liệu này chỉ mô phỏng vùng Tây Nguyên (Đắk Lắk, Lâm Đồng). Nếu áp dụng cho vùng khác, model có thể bị sai lệch do khác biệt khí hậu và giá tại vườn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("../data/raw/mock_coffee_data.csv")
df["date"] = pd.to_datetime(df["date"])
print(f"Shape: {df.shape}")
df.head()

## 1. Thống kê mô tả

In [ ]:
# Thống kê mô tả + số lượng NaN mỗi cột
desc = df.describe().T
desc["na_count"] = df.isna().sum()
desc["na_pct"] = (df.isna().sum() / len(df)) * 100
desc

## 2. Ma trận tương quan (Correlation Heatmap)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix: Thời tiết vs Giá cà phê")
plt.tight_layout()
plt.show()

## 3. Time Series — Giá cà phê theo thời gian

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["date"], df["historical_price_vnd"], alpha=0.7)
plt.title("Giá cà phê theo thời gian (Mock Data)")
plt.xlabel("Ngày")
plt.ylabel("Giá (VND/kg)")
plt.tight_layout()
plt.show()

## 4. Phân bố giá theo tháng (mùa vụ)

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(x="month", y="historical_price_vnd", data=df)
plt.title("Phân bố giá cà phê theo tháng — Mùa vụ thu hoạch (11-3) cao hơn")
plt.xlabel("Tháng")
plt.ylabel("Giá (VND/kg)")
plt.tight_layout()
plt.show()

## 5. Phát hiện Outliers (Boxplot)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cols = ["avg_temp_c", "rainfall_mm", "humidity_pct", "sunshine_hours"]
for ax, col in zip(axes.flat, cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.suptitle("Outliers trong đặc trưng thời tiết")
plt.tight_layout()
plt.show()

## 6. Phân bố giá cà phê

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["historical_price_vnd"], kde=True, bins=50)
plt.title("Phân bố giá cà phê (VND/kg)")
plt.xlabel("Giá (VND/kg)")
plt.ylabel("Tần suất")
plt.tight_layout()
plt.show()

## 7. Scatter: Nhiệt độ vs Giá

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x="avg_temp_c", y="historical_price_vnd", hue="month", palette="viridis", data=df, alpha=0.6)
plt.title("Nhiệt độ vs Giá cà phê (màu = tháng)")
plt.xlabel("Nhiệt độ TB (°C)")
plt.ylabel("Giá (VND/kg)")
plt.tight_layout()
plt.show()

## 8. Missing Data Pattern

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
if len(missing):
    missing.plot(kind="bar", color="salmon")
    plt.title("Số lượng giá trị thiếu theo cột")
    plt.ylabel("Số NaN")
    plt.tight_layout()
    plt.show()
else:
    print("Không có NaN trong dataset.")

## Kết luận & Ghi chú

- Dữ liệu có seasonal pattern rõ ràng (tháng 11–3 giá cao hơn).
- Outliers được inject ~2% để test pipeline.
- NaN ~5% — pipeline tiền xử lý cần xử lý missing values.
- **Bias Limitation:** Data chỉ đại diện cho Tây Nguyên.